# 05 — Full CAP-ZW Training

This notebook trains the proposed collision-aware, Pareto-optimized zero-watermark representation. It uses attack-paired images and optimizes robustness, hard-negative discrimination, bit balance, entropy, and decorrelation. No reported paper number is hard-coded.


In [ ]:
from pathlib import Path
import pandas as pd, torch
from torch.utils.data import DataLoader
from zero_watermarking.datasets import load_manifest, validate_manifest, group_split
from zero_watermarking.torch_data import AttackPairDataset
from zero_watermarking.learned import HashEncoder
from zero_watermarking.training import TrainConfig, train_cap_zw


In [ ]:
MANIFEST = Path('../data/manifests/medical_manifest.csv')
if not MANIFEST.exists():
    raise FileNotFoundError('Create a real medical manifest first; see data/README.md')
frame = validate_manifest(load_manifest(MANIFEST))
train, val, test = group_split(frame, seed=42)
records = load_manifest(MANIFEST)
train_ids = set(train.image_id)
train_records = [r for r in records if r.image_id in train_ids]
dataset = AttackPairDataset(train_records, attack_name='gaussian_noise', attack_param=0.03)
loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)
model = HashEncoder(nbits=256, use_bemq=True)
history = train_cap_zw(model, loader, TrainConfig(epochs=10), checkpoint='../experiments/results/cap_zw.pt')
pd.DataFrame(history)

## Required reporting
After training, evaluate on the held-out test groups with the full attack-strength grid and report confidence intervals. Do not tune thresholds or attack choices on the test set.